# FluxIA — IA e semáforos inteligentes no SUMO

**FIAP Global Solution CitizenTech 2026 · entrega de Inteligência Artificial e Machine Learning**

**Autores:** Wesley Zimmer (RM 560264) e João Victor Alvarenga (RM 93752).

Este notebook apresenta problema e dados, preparação, modelo, treinamento, avaliação e integração. O ZIP contém os dados, código-fonte, rede SUMO e checkpoint.

> **Escopo:** recorte central com 8 semáforos controlados em conjunto. Os carros usam rotas sintéticas. A CET traz registros de lentidão por trecho, não contagem de veículos. O protótipo não controla semáforos reais.


## Critérios da rubrica

| Critério | Conteúdo |
|---|---|
| Problema e dados (2,0) | Congestionamento e análise da base CET |
| Preparação e análise (2,0) | Limpeza, recorte, prioridades espaciais e cenários |
| Modelo e treinamento (3,0) | PPO, observações, ações, recompensa, parâmetros e histórico |
| Avaliação com métrica (2,0) | Cinco sementes e três intensidades no conjunto de validação |
| Integração na solução (1,0) | Política conectada ao SUMO por TraCI |


## 1. Problema, recorte e dados

Uma fila em um cruzamento pode bloquear a saída e dificultar o avanço das ruas conectadas. A política recebe observações dos oito semáforos e escolhe ações em conjunto, buscando reduzir atrasos e filas enquanto o ambiente protege as transições de fase.

A base da CET é de 2023. O recorte CENTRO contém **38.113 registros**; 18h tem o maior número de registros (**3.175**), seguido de 17h (**3.110**). Esses registros são observações de lentidão, não carros contados.

Fonte: [Base de dados sobre lentidão por trechos — CET / Prefeitura de São Paulo](https://dados.prefeitura.sp.gov.br/dataset/base-de-dados-sobre-lentidao-por-trechos-cet).

In [ ]:
from pathlib import Path
import json, os, sys
ROOT=next((p for p in (Path.cwd(),*Path.cwd().parents)
           if (p/"scripts"/"train_fluxia_ppo.py").is_file()),None)
if ROOT is None:
    raise FileNotFoundError("Abra o notebook dentro da pasta extraída entrega-ia-ml.")
os.chdir(ROOT)
print("Pasta portátil:",ROOT.name)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
cet=pd.read_csv(ROOT/"data/raw/cet_lentidao_trechos_2023.csv",sep=";",encoding="cp1252")
cet["data"]=pd.to_datetime(cet["data"],format="%d/%m/%Y %H:%M",errors="coerce")
cet["tamanho"]=pd.to_numeric(cet["tamanho"],errors="coerce")
cet["nome_regiao"]=cet["nome_regiao"].astype(str).str.strip().str.upper()
centro=cet.loc[cet["nome_regiao"].eq("CENTRO")].dropna(subset=["data"]).copy()
print(f"Base completa: {len(cet):,}; recorte CENTRO: {len(centro):,} registros.")
hourly=(centro.assign(hora=centro["data"].dt.hour).groupby("hora",as_index=False)
        .agg(registros=("data","size"),metros_registrados=("tamanho","sum")))
display(hourly.sort_values("registros",ascending=False).head(8))
ax=hourly.set_index("hora")["registros"].plot(kind="bar",figsize=(10,4),color="#1769aa")
ax.set(title="Registros de lentidão no CENTRO por hora — CET 2023",
       xlabel="Hora do dia",ylabel="Registros")
plt.tight_layout()
plt.show()


### Como a CET é usada

Horários e corredores com mais registros ajudam a priorizar trechos para as rotas. Uma correspondência manual entre os nomes da CET e as vias do OpenStreetMap os liga às arestas da rede SUMO; cada relação tem um nível de confiança e não é uma medição de fluxo.

As rotas dos veículos no SUMO são sintéticas. A base da CET não contém trajetórias individuais de carros.

In [ ]:
hotspots=json.loads((ROOT/"data/processed/cet_hotspots_centro_pm_2023.json")
                     .read_text(encoding="utf-8"))
print("Método:",hotspots.get("spatial_method","ver arquivo de dados processados"))
display(pd.json_normalize(hotspots.get("hotspots",[])).head(10))


## 2. Preparação dos cenários de tráfego

O conjunto congelado separa rotas em treino, validação e teste, com intensidades moderada, pico e estresse. As rotas são sintéticas e verificadas contra a rede C2 de oito semáforos. A validação não usou o conjunto de teste; as rotas de teste não estão no ZIP.

| Divisão | Pares origem-destino | Definições de rota |
|---|---:|---:|
| Treino | 360 | 611 |
| Validação | 120 | 223 |
| Teste | 120 | 219 |

Os mesmos pares são usados nos três perfis, com volumes programados diferentes. Estresse é uma demanda simulada intensa, não um dia real da CET.

In [ ]:
scenario_summary=json.loads((ROOT/"evidence/cenario_resumo_portatil.json").read_text(encoding="utf-8"))
scenario_table=pd.DataFrame(scenario_summary["cenarios"])
display(scenario_table[["split","intensidade","ods","definicoes_de_rota","veiculos_programados","valido_na_rede_c2"]])


## 3. Modelo, ações e recompensa

Usamos **MaskablePPO** de sb3-contrib. É uma política conjunta: uma decisão contém uma ação para cada um dos oito semáforos. A observação tem formato (8, 16) e cada semáforo dispõe de seis ações. Máscaras retiram ações que não são legais no estado atual.

As observações incluem fase e tempo de fase, fila e velocidade locais, capacidade observada ou prevista a jusante, estado de cruzamentos vizinhos e idade dos dados. O ambiente valida a mudança de fase e protege a sequência amarela e a limpeza totalmente vermelha.

| Parte da recompensa | Efeito |
|---|---|
| Atraso e tempo perdido | Penaliza tempo de viagem perdido |
| Fila e espera excessiva | Penaliza veículos esperando |
| Bloqueio a jusante | Penaliza receptor cheio |
| Chegadas ao destino | Recompensa veículos que completam a rota |
| Demanda pendente | Penaliza veículos ainda não inseridos no prazo |
| Colisão e teletransporte | Penaliza ocorrências do SUMO |

A recompensa orienta o aprendizado, mas não demonstra melhora. A avaliação usa também tempo de viagem, espera, chegadas, pendências e ocorrências.

### O que a política pode pedir e como a recompensa é calculada

As seis ações por sinal correspondem a manter o plano atual, solicitar amarelo, estender o verde dentro do limite ou solicitar uma das fases verdes de destino. O ambiente mantém o amarelo com a duração configurada e verifica a limpeza totalmente vermelha antes de liberar a próxima fase.

| Componente implementado | Peso no retorno |
|---|---:|
| Atraso / tempo perdido | -0,05 por segundo de perda |
| Fila | -0,02 por segundo-veículo, normalizado pelos sinais |
| Receptor a jusante cheio | -0,50 por sinal sem capacidade, normalizado |
| Veículo que chega | +0,30 por chegada |
| Espera excessiva | -0,05 por sinal, normalizado |
| Veículo devido ainda não inserido | -0,05 por sinal, normalizado |
| Espera em vermelho total | -0,10 por sinal retido |
| Colisão no SUMO | -20 por passo-veículo em colisão |
| Teletransporte pelo SUMO | -5 por veículo removido |

Esses valores são os pesos usados no ambiente de treinamento. Eles são escolhas experimentais do projeto, não valores oficiais da CET.

In [ ]:
manifest=json.loads((ROOT/"evidence/training_manifest_portatil.json").read_text(encoding="utf-8"))
print("Algoritmo:",manifest["algorithm"])
print("Política:",manifest["policy"])
print("Observação:",manifest["observation_shape"])
print("Ações por sinal:",manifest["action_count_per_signal"])
print("Passos pedidos:",manifest["requested_timesteps"])
print("Passos do checkpoint:",manifest["checkpoint_internal_timesteps"])
print("Treino chegou ao total pedido:",manifest["training_reached_requested_timesteps"])
print("Parâmetros:",manifest["hyperparameters"])


### Histórico do treinamento

O treino pediu **100.608 passos**, com seis ambientes SUMO paralelos e GPU. O checkpoint escolhido foi salvo no marco de 46.080 passos; o modelo registra 46.084 passos internos. O treino não atingiu o total solicitado. A curva abaixo vem do histórico salvo. Treinar novamente não faz parte do Run All.

In [ ]:
progress=pd.read_csv(ROOT/"evidence/training_progress.csv")
x,y="time/total_timesteps","rollout/ep_rew_mean"
curve=progress[[x,y]].apply(pd.to_numeric,errors="coerce").dropna()
ax=curve.plot(x=x,y=y,figsize=(9,4),legend=False,color="#7a3db8")
ax.axvline(46080,color="#d95f02",linestyle="--",label="Checkpoint escolhido")
ax.set(title="Retorno médio registrado durante o treinamento",xlabel="Passos",ylabel="Retorno médio por episódio")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
from sb3_contrib import MaskablePPO
CHECKPOINT=ROOT/"models/fluxia_ppo_8signals_46084.zip"
model=MaskablePPO.load(CHECKPOINT,device="cpu")
print("Modelo carregado em CPU.")
print("Passos:",model.num_timesteps)
print("Formato da observação:",model.observation_space.shape)
print("Ações por semáforo:",model.action_space.nvec)


## 4. Avaliação

O checkpoint foi comparado em cinco sementes por perfil, com episódios de 600 segundos. Os dados vêm do conjunto de validação; o conjunto de teste permaneceu intocado.

A validação atual não mostra melhora consistente em relação ao PPO anterior. No perfil de estresse, o tempo médio ficou ligeiramente maior e as ocorrências não desapareceram. O modelo foi treinado e integrado ao simulador, mas **um ganho de trânsito devido ao treinamento adicional não está demonstrado**.

### Médias da validação registrada (cinco sementes por perfil)

| Perfil | Controle | Chegadas | Espera média (s) | Viagem média (s) | Pendentes | Colisões | Teletransportes |
|---|---|---:|---:|---:|---:|---:|---:|
| Moderado | Plano fixo nativo | 63,2 | 73,31 | 211,70 | 0,0 | 0,0 | 0,0 |
| Moderado | PPO anterior | 64,0 | 69,94 | 207,51 | 0,0 | 0,0 | 0,0 |
| Moderado | PPO escolhido (~46k) | 64,2 | 70,09 | 207,64 | 0,0 | 0,0 | 0,0 |
| Pico | Plano fixo nativo | 122,4 | 78,25 | 227,44 | 0,0 | 0,2 | 0,2 |
| Pico | PPO anterior | 122,6 | 78,61 | 227,71 | 0,0 | 0,2 | 0,2 |
| Pico | PPO escolhido (~46k) | 122,6 | 78,61 | 227,67 | 0,0 | 0,0 | 0,0 |
| Estresse | Plano fixo nativo | 218,6 | 82,00 | 235,85 | 6,2 | 0,2 | 0,2 |
| Estresse | PPO anterior | 219,0 | 79,43 | 232,96 | 7,0 | 0,6 | 0,6 |
| Estresse | PPO escolhido (~46k) | 219,0 | 79,58 | 233,10 | 7,0 | 0,8 | 0,8 |

No perfil de estresse, a média de tempo do checkpoint escolhido é cerca de 0,15 s maior que a do PPO anterior e suas médias de colisões e teletransportes são 0,2 maiores. Esse resultado não mostra ganho consistente com os passos extras de treinamento.

In [ ]:
import statistics
validation=json.loads((ROOT/"evidence/paired_validation_summary.json").read_text(encoding="utf-8"))
assert validation["validation_only"] and not validation["test_set_touched"]
rows=[]
for profile,result in validation["profiles"].items():
    runs=result["checkpoint_46080_runs"]
    prev=result["previous_validation_means"]
    current={
      "arrivals_mean":statistics.mean(r["arrivals"] for r in runs),
      "mean_wait_s_mean":statistics.mean(r["mean_wait_s"] for r in runs),
      "mean_duration_s_mean":statistics.mean(r["mean_duration_s"] for r in runs),
      "pending_mean":statistics.mean(r["pending"] for r in runs),
      "collisions_mean":statistics.mean(r["collisions_xml"] for r in runs),
      "teleports_mean":statistics.mean(r["teleports"] for r in runs)}
    for label,v in [("Plano fixo nativo",prev["native"]),("PPO anterior",prev["ppo"]),
                    ("PPO escolhido (~46k)",current)]:
        rows.append({"perfil":profile,"controle":label,"chegadas":v["arrivals_mean"],
          "espera média (s)":v["mean_wait_s_mean"],"viagem média (s)":v["mean_duration_s_mean"],
          "pendentes":v["pending_mean"],"colisões":v["collisions_mean"],
          "teletransportes":v["teleports_mean"]})
metrics=pd.DataFrame(rows)
display(metrics.round(2))


Chegadas contam veículos que completaram a rota em 600 segundos. Tempo de viagem e espera são médias dos veículos que chegaram. Veículos ativos e pendentes ficam separados; colisões e teletransportes vêm do SUMO. Poucas sementes e diferenças pequenas não provam melhoria geral.

## 5. Integração da política com SUMO

O SUMO atualiza a rede; o ambiente monta observações dos oito sinais; o PPO escolhe ações; TraCI aplica as mudanças após as proteções de fase. A validação registrada para estresse durou 600 segundos: **218 chegadas**, **377 veículos ativos no fim**, **6 pendentes**, **1 colisão** e **1 teletransporte**.

Isto comprova integração local com o simulador, não com câmeras reais nem com semáforos da rua. A célula abaixo usa a evidência salva. Altere EXECUTAR_NOVA_SIMULACAO para True para rodar outro episódio.

In [ ]:
EXECUTAR_NOVA_SIMULACAO=False  # True para iniciar novo episódio SUMO de 600 s
if EXECUTAR_NOVA_SIMULACAO:
    import subprocess
    out=ROOT/"outputs/evaluation_reproduced"
    out.mkdir(parents=True,exist_ok=True)
    cmd=[sys.executable,str(ROOT/"scripts/train_fluxia_ppo.py"),"evaluate",
      "--network",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/centro_fluxia_c2.net.xml"),
      "--catalog",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/pilot_signals_8_c2.json"),
      "--timing-profile",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/green_timing_profile_c2.json"),
      "--routes",str(ROOT/"docs/execucao-luna/inputs/c5/frozen_scenarios/validation/estresse/demand.rou.xml"),
      "--checkpoint",str(CHECKPOINT),"--output",str(out),"--episode-output",str(out/"episodes"),
      "--seed","20261301","--episode-horizon","600","--device","cpu"]
    run=subprocess.run(cmd,cwd=ROOT,text=True,capture_output=True)
    if run.returncode:
        print(run.stdout); print(run.stderr)
        raise RuntimeError("Avaliação SUMO falhou. Confira a instalação descrita no README.")
    result_path=out/"evaluation.json"
else:
    result_path=ROOT/"evidence/integration_evaluation_seed_20261301.json"
integration=json.loads(result_path.read_text(encoding="utf-8"))
print("Origem:",integration.get("evidence_origin","nova execução"))
print("Tempo:",integration["simulation_time_s"],"s; chegadas:",integration["arrivals"])
print("Ativos no fim:",integration["active_at_end"],"; pendentes:",integration["due_pending_at_end"])
print("Recompensa:",round(integration["reward_total"],2))


## Treinamento opcional do zero

O treinamento completo não roda no Run All. A célula abaixo inicia manualmente treino em CPU se TREINAR_DO_ZERO for alterado para True; pode levar bastante tempo. O checkpoint pronto está incluído.

In [ ]:
TREINAR_DO_ZERO=False
if TREINAR_DO_ZERO:
    import subprocess
    routes=[ROOT/f"docs/execucao-luna/inputs/c5/frozen_scenarios/training/{p}/demand.rou.xml"
            for p in ("moderado","pico","estresse")]
    cmd=[sys.executable,str(ROOT/"scripts/train_fluxia_ppo.py"),"train",
      "--network",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/centro_fluxia_c2.net.xml"),
      "--catalog",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/pilot_signals_8_c2.json"),
      "--routes",str(routes[0]),"--training-routes",*[str(p) for p in routes],
      "--timing-profile",str(ROOT/"sumo/scenarios/fluxia_stage6_c2/green_timing_profile_c2.json"),
      "--output",str(ROOT/"outputs/training_new"),"--episode-output",str(ROOT/"outputs/training_new/episodes"),
      "--seed","20261119","--n-envs","1","--total-timesteps","100608",
      "--episode-horizon","8100","--device","cpu"]
    subprocess.run(cmd,cwd=ROOT,check=True)


## Conclusão e limites

O protótipo conecta registros agregados de lentidão, rotas sintéticas, um PPO conjunto para oito semáforos e SUMO. Os resultados atuais **não provam que treinar por mais tempo melhora o trânsito de forma consistente**. A CET é de 2023; a demanda é simulada; o recorte não representa toda São Paulo e não usa câmeras nem semáforos reais.

### Referências

- CET / Prefeitura de São Paulo: [Base de lentidão por trechos](https://dados.prefeitura.sp.gov.br/dataset/base-de-dados-sobre-lentidao-por-trechos-cet)
- SUMO: [Documentação oficial](https://sumo.dlr.de/docs/)
- Os arquivos de rede e cenários usados acompanham este ZIP.
